# 01 – Stream processing avec Spark Structured Streaming
Kafka `reviews_stream` → parsing JSON → filtres / fenêtres / statistiques → PostgreSQL.
Le producteur doit tourner (`docker compose up -d producer`).

In [ ]:
import time
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               BooleanType, ArrayType)
from shopstream_utils import (get_spark, read_table, write_table,
                              KAFKA_BOOTSTRAP, TOPIC, CHECKPOINT_DIR)

spark = get_spark("01-streaming", shuffle_partitions=4)

## 2.1 Lire le topic Kafka

In [ ]:
# TODO : DataFrame de streaming sur le topic Kafka (startingOffsets=earliest, maxOffsetsPerTrigger=500)
raw = ...
raw.printSchema()

## 2.2 Parser et nettoyer

In [ ]:
# TODO : schéma explicite de la valeur JSON (attention à hashtags : tableau de chaînes)
review_schema = StructType([
    StructField("review_id", StringType()),
    # ...
])

# TODO : value -> string -> from_json -> colonnes à plat
#        + event_time en timestamp, text_length, kafka_partition, kafka_offset, ingest_time
#        + filtrer les lignes invalides
parsed = ...
parsed.printSchema()

## 2.3 Transformation 1 – filtrage (langue + mot-clé)
Aperçu avec le sink `memory` (pratique dans un notebook).

In [ ]:
# TODO : avis FR uniquement + colonne booléenne mentions_delivery (livraison|delivery, insensible à la casse)
reviews_fr = ...

# Aperçu : sink memory, attendre ~20 s, interroger la table en SQL, puis ARRÊTER la requête

## 2.4 Sink 1 – avis bruts → PostgreSQL `reviews_stream`
Démo : `docker compose exec postgres psql -U spark -d shopstream` puis `SELECT count(*) FROM reviews_stream;` et `\watch 2`.

In [ ]:
# TODO : foreachBatch -> table reviews_stream (append), trigger 10 s, checkpoint dédié
def write_raw(batch_df, batch_id):
    ...

q_raw = ...

## 2.5 Transformation 2 + Action 1 – fenêtres d'1 minute par catégorie (watermark 2 min, mode append)

In [ ]:
# TODO : fenêtre fixe d'1 minute par catégorie, watermark 2 minutes, count + note moyenne
#        colonnes finales : window_start, window_end, category, nb_reviews, avg_rating
windowed = ...

# TODO : écriture en mode append dans stream_window_counts (foreachBatch + checkpoint dédié)
q_win = ...

### Q2.2 – même agrégation en mode `update` vers la console (debug)

In [ ]:
# TODO Q2.2 : même agrégation (windowed) vers le sink console en mode update, ~25 s, puis stop()
# (la sortie console apparaît dans : docker compose logs -f jupyter)

## 2.6 Action 2 – statistiques par hashtag (mode complete, table écrasée à chaque micro-batch)

In [ ]:
# TODO : explode(hashtags) puis par hashtag : nb_reviews, avg_text_length, avg_rating
hashtag_stats = ...

# TODO : mode complete + foreachBatch qui écrase stream_hashtag_stats
q_tags = ...

## 2.7 Supervision

In [ ]:
# TODO : pour chaque requête active, afficher name, status et les indicateurs de lastProgress

In [ ]:
for t in ["reviews_stream", "stream_window_counts", "stream_hashtag_stats"]:
    print(t, read_table(spark, t).count())
read_table(spark, "stream_window_counts").orderBy(F.desc("window_start")).show(8)
read_table(spark, "stream_hashtag_stats").orderBy(F.desc("nb_reviews")).show(8)

### Reprise sur checkpoint et contrôle des doublons

In [ ]:
# TODO : arrêter q_raw, attendre, la relancer avec LE MÊME checkpoint, puis compter les doublons de review_id

**Laissez `q_raw` tourner** pendant les parties 3 et 4. En fin de journée :
```python
for q in spark.streams.active: q.stop()
```

## Réponses aux questions
- **Q1.1** : ...
- **Q1.2** : ...
- **Q1.3** : ...
- **Q2.1** : ...
- **Q2.2** : ...
- **Q2.3** : ...
- **Q2.4** : ...